In [22]:
from typing import Annotated
from fastapi import FastAPI,Query
from pydantic import AfterValidator

app = FastAPI()


@app.get("/items/")
def read_items(q:str | None = None):
    results = {"items":[{"item_id":"Foo"},{"item_id":"Bar"}]}
    if q:
        results.update({"q":q})
    return results



In [8]:
# 额外校验,该参数提供时，长度不能超过50
@app.get("/items/")
def read_items(q:Annotated[str | None,Query(min_length=5,max_length=50)] = None):
    results = {"items":[{"item_id":"Foo"},{"item_id":"Bar"}]}
    if q:
        results.update({"q":q})
    return results

In [9]:
# 正则表达式,必须是fixedquery值
@app.get("/items/")
def read_items(q:Annotated[str | None,Query(min_length=5,max_length=50,pattern="^fixedquery$")] = None):
    results = {"items":[{"item_id":"Foo"},{"item_id":"Bar"}]}
    if q:
        results.update({"q":q})
    return results

In [10]:
# 接收多个值

@app.get("/items/")
def read_items(q:Annotated[list[str] | None,Query()]=None):
    query_items = {"q":q}
    return query_items

In [13]:
#具有默认值的查询参数，多个值

@app.get("/items")
def read_items(q:Annotated[list[str],Query()] = ['foo','bar']):
    query_items = {"q":q}
    return query_items

In [14]:
# 添加title
@app.get("/items/")
def read_items(
    q:Annotated[str | None,Query(title="Query String",min_length=3)] = None
):
    results = {"items":[{"item_id":"foo"},{"item_id":"bar"}]}
    if q:
        results.update({"q":q})
    return results

In [15]:
# 添加注释
@app.get("/items/")
def read_items(
    q:Annotated[str | None,Query(description='Query string for the items to search in the database that have a good match',
                                 min_length=3)] = None
):
    results = {"items":[{"item_id":"foo"},{"item_id":"bar"}]}
    if q:
        results.update({"q":q})
        return results

In [ ]:
# 别名 
@app.get("/items/")
def read_items(
    q:Annotated[str |None,Query(alias='item-query')] = None
):
    results = {"items":[{"item_id":"foo"},{"item_id":"bar"}]}
    if q:
        results.update({"q":q})
    return results
 

In [ ]:
# 弃用参数 True 而不是true
@app.get("/items/")
def read_items(
    q:Annotated[str |None ,Query(deprecated=True)] = None
):
    results = {"items":[{"item_id":"foo"},{"item_id":"bar"}]}
    if q:
        results.update({"q":q})
    return results 

In [21]:
# 隐藏某个查询参数
@app.get("/items/")
def read_items(
    hidden_query: Annotated[str | None,Query(include_in_schema=False)] = None,
    q:str | None = None
):
    if hidden_query:
        return {"hidden_query":hidden_query}
    else:
        return {"hidden_query":"Not Found"}

In [24]:
# 自定义参数校验

data = {
    "isbn-9781529046137": "The Hitchhiker's Guide to the Galaxy",
    "imdb-tt0371724": "The Hitchhiker's Guide to the Galaxy",
    "isbn-9781439512982": "Isaac Asimov: The Complete Stories, Vol. 2",
}

def check_valid_id(id:str):
    if not id.startswith('isbn-','imdb-'):
        raise ValueError('Invalid Id format,it must start with "isbn-" or "imdb-')
    return id


@app.get("/items/")
def read_items(id :Annotated[str | None,AfterValidator(check_valid_id)]):
    if id:
        item = data.get(id)
    else:
        id,item= random.choice(list[data.items])
    return {"id":id,"name":item}